## What this notebook establishes

- A force push is **refused outright**: no confirmation is offered, the tool result is an error, and no file appears.
- An ordinary `git push`, or anything mentioning `migrations/`, is a **question**: one confirmation is shown, and the answer decides whether the command runs.
- With **no UI to ask**, the confirming half takes the safe branch and blocks. A headless run cannot approve anything.
- The retained pre-fix guard shows the contrast on the same scenario: it *asks* about a force push, and a person who says yes lets it through.

## What this notebook does **not** establish

- **A guard is a procedure, not containment.** `metadata/11-chapter.yaml` records that the four rules, the four questions and the placement table are **proposed** by this book. Chapter 42 is about the half that must *hold*.
- **The procedure assumes the need is classifiable.** A command that means `git push` without containing it will not match, and the chapter says so.
- **The commands really run.** `git push` fails here because the temporary directory is not a repository; that is the *tool's* honest failure and it is what makes the effect check meaningful — the flag file is created before the push in every allowed case.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — the chapter's own guard extension loaded unmodified into real sessions, plus the retained pre-fix version as a control.
* **Evidence scope:** `in_process_runtime`. The gate is the extension's; the tool is Pi's `bash`; the model is scripted.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(11))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/11-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: one extension, two kinds of rule

`ch11-guard/guard.ts` is 16 lines and contains both halves:

* a **prohibition** — a force push matches a list and returns `{ block: true }` with a
  reason, with no question asked;
* a **confirmation** — anything else matching `git push` or `migrations/` calls
  `ctx.ui.confirm`, and the answer decides.

The driver runs each case in its own session and reports three things: how many
confirmations were shown, whether the tool result was an error, and whether a file
the command would create actually exists.

In [3]:
GUARD = pinb.driver_source("ch11-guard.ts")
guard = pinb.driver(GUARD, label="ch11-guard", timeout=420)
print(pinb.md_table(
    ["case", "confirmations shown", "tool result", "ran.flag created?"],
    [[guard[k]["label"], guard[k]["confirmsShown"],
      "error" if guard[k]["results"][0]["isError"] else "ok",
      "YES" if guard[k]["ran"][0] else "no"] for k in guard],
))
print()
print("The reason the model received, in the two blocked cases:")
print("  no UI      :", guard["noUI"]["results"][0]["text"])
print("  person says no:", guard["ordinaryRefused"]["results"][0]["text"])
print("  force push :", guard["forcePush"]["results"][0]["text"])

| case | confirmations shown | tool result | ran.flag created? |
|---|---|---|---|
| no UI to ask | 0 | error | no |
| force push, person says yes | 0 | error | no |
| ordinary push, person says yes | 1 | error | YES |
| ordinary push, person says no | 1 | error | no |
| a migrations/ command, person says yes | 1 | error | YES |
| an unrelated command | 0 | ok | YES |
| the first version of this guard, a force push | 1 | error | YES |
| the first version, an ordinary push | 1 | error | YES |

The reason the model received, in the two blocked cases:
  no UI      : Force pushes are not permitted by this repository's guard
  person says no: Blocked by guard extension
  force push : Force pushes are not permitted by this repository's guard


In [4]:
pinb.show_checks([
    pinb.check("a force push is refused with no question asked",
               guard["forcePush"]["confirmsShown"] == 0 and guard["forcePush"]["results"][0]["isError"],
               "0 confirmations, isError"),
    pinb.check("and nothing ran", not guard["forcePush"]["ran"][0], "ran.flag absent"),
    pinb.check("an ordinary push asks exactly once", guard["ordinary"]["confirmsShown"] == 1, "1 confirmation"),
    pinb.check("saying yes lets it run", guard["ordinary"]["ran"][0] is True, "ran.flag present"),
    pinb.check("saying no blocks it and nothing runs",
               guard["ordinaryRefused"]["results"][0]["isError"] and not guard["ordinaryRefused"]["ran"][0],
               guard["ordinaryRefused"]["results"][0]["text"]),
    pinb.check("a migrations/ path is treated the same way",
               guard["migrations"]["confirmsShown"] == 1 and guard["migrations"]["ran"][0] is True,
               "asked once, then ran"),
    pinb.check("an unrelated command is left alone",
               guard["quiet"]["confirmsShown"] == 0 and not guard["quiet"]["results"][0]["isError"],
               "no question, ran cleanly"),
    pinb.check("with no UI the confirming half blocks rather than allowing",
               guard["noUI"]["confirmsShown"] == 0 and not guard["noUI"]["ran"][0],
               "0 confirmations, nothing ran"),
])

== Assertions ==
  assertion  observed
----  ----------------------------------------------------------  ----------------------------
PASS  a force push is refused with no question asked  0 confirmations, isError
PASS  and nothing ran  ran.flag absent
PASS  an ordinary push asks exactly once  1 confirmation
PASS  saying yes lets it run  ran.flag present
PASS  saying no blocks it and nothing runs  Blocked by guard extension
PASS  a migrations/ path is treated the same way  asked once, then ran
PASS  an unrelated command is left alone  no question, ran cleanly
PASS  with no UI the confirming half blocks rather than allowing  0 confirmations, nothing ran

8/8 assertions held.


## Contrasting case: the guard before it was fixed

The repository keeps `ch11-guard/guard-original.ts` for exactly this reason — the
first version treated a force push as a question. The two rows below are the same
scenario under the two guards.

The difference is not the pattern; it is the **kind of rule**. A prohibition with a
question attached is not a prohibition, because a person can say yes.

In [5]:
print(pinb.md_table(
    ["guard", "scenario", "confirmations", "ran.flag created?"],
    [["current", "a force push", guard["forcePush"]["confirmsShown"], "YES" if guard["forcePush"]["ran"][0] else "no"],
     ["pre-fix", "a force push", guard["originalForcePush"]["confirmsShown"], "YES" if guard["originalForcePush"]["ran"][0] else "no"],
     ["current", "an ordinary push", guard["ordinary"]["confirmsShown"], "YES" if guard["ordinary"]["ran"][0] else "no"],
     ["pre-fix", "an ordinary push", guard["originalOrdinary"]["confirmsShown"], "YES" if guard["originalOrdinary"]["ran"][0] else "no"]],
))

pinb.show_checks([
    pinb.check("the current guard refuses a force push", not guard["forcePush"]["ran"][0], "refused"),
    pinb.check("the pre-fix guard lets a person approve one", guard["originalForcePush"]["ran"][0] is True,
               "approved, and the command ran"),
    pinb.check("both guards still ask about an ordinary push",
               guard["ordinary"]["confirmsShown"] == 1 and guard["originalOrdinary"]["confirmsShown"] == 1,
               "1 confirmation each"),
])

| guard | scenario | confirmations | ran.flag created? |
|---|---|---|---|
| current | a force push | 0 | no |
| pre-fix | a force push | 1 | YES |
| current | an ordinary push | 1 | YES |
| pre-fix | an ordinary push | 1 | YES |
== Assertions ==
  assertion  observed
----  --------------------------------------------  -----------------------------
PASS  the current guard refuses a force push  refused
PASS  the pre-fix guard lets a person approve one  approved, and the command ran
PASS  both guards still ask about an ordinary push  1 confirmation each

3/3 assertions held.


## The chapter's own tests

The driver above is built from the chapter's guard; the assertion set that belongs
to the chapter is its test file, which also covers the "no UI to ask" branch and the
migrations half directly.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [6]:
PATTERNS = 'ch11-guard/guard.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch11-guard/guard.test.ts', show="guard")

$ node --test ch11-guard/guard.test.ts
  PASS  6 passed, 0 failed, 0 skipped  in 1209 ms

  tests matching 'guard':
    ok   regression: the first version of this guard missed a plain `git push`
    ok   the fixed guard asks about a plain git push, and a person can allow or refuse


## Your turn: predict, then run

**Predict first.** Add a third kind of rule to the guard: anything touching
`production` is refused outright, like a force push. Which existing case would that
have to be distinguished from — and what happens to the `git push` confirmation if
you get the order wrong?

**Then change one input.** `COMMAND` below is what the model asks for. Run it with
a force push, then with `git push`, then with an unrelated command.

**Predict the boundary.** The force-push pattern is matched against
`JSON.stringify(event.input)`, not against the raw argument. What would a command
that *contains the words* `git push --force` inside a quoted string do — block, or
let through?

In [7]:
COMMAND = "git push origin main"

print("Current pattern:", r"/\bgit\s+push\b.*(?:\s--force\b|\s-f\b|--force-with-lease\b)/")
print("Matched against JSON.stringify(event.input), not the raw argument.")
print()

cases = {
    "git push --force origin main": True,
    "git push -f origin main": True,
    "git push --force-with-lease origin main": True,
    "git push origin main": False,
    'echo "remember to git push --force later"': True,
    "git  push   --force": True,
}
pattern = __import__("re").compile(r"\bgit\s+push\b.*(?:\s--force\b|\s-f\b|--force-with-lease\b)")
print("Predict, then check, for each command:")
for command, expected in cases.items():
    matched = bool(pattern.search(__import__("json").dumps({"command": command})))
    print(f"  {command!r:52s} blocked={matched}")
print()
print("The last row is the answer to the boundary question: the pattern matches the")
print("serialised input, so a *mention* of a force push inside a quoted string")
print("blocks a command that was never going to force push. Refusing is the safe")
print("direction; allowing would not be.")

Current pattern: /\bgit\s+push\b.*(?:\s--force\b|\s-f\b|--force-with-lease\b)/
Matched against JSON.stringify(event.input), not the raw argument.

Predict, then check, for each command:
  'git push --force origin main'                       blocked=True
  'git push -f origin main'                            blocked=True
  'git push --force-with-lease origin main'            blocked=True
  'git push origin main'                               blocked=False
  'echo "remember to git push --force later"'          blocked=True
  'git  push   --force'                                blocked=True

The last row is the answer to the boundary question: the pattern matches the
serialised input, so a *mention* of a force push inside a quoted string
blocks a command that was never going to force push. Refusing is the safe
direction; allowing would not be.


## Interpretation

The chapter's four ways to change Pi are a spectrum, and this notebook puts a number
on the difference between the two ends:

| Kind of rule | Where it lives | Can a person override it? | Does it hold without a person? |
|---|---|---|---|
| **prohibition** | the extension's own code | no — there is nothing to approve | **yes**, including headless |
| **confirmation** | `ctx.ui.confirm` | yes, by answering | no — with no UI it must pick, and the safe pick is to block |
| instruction (chapters 12–14) | files the model reads | yes, by editing the file | no — it is prose |
| the model's own judgement | nowhere | n/a | no |

A guard that is only the second row is a speed bump. The value of the first row is
that it does not depend on anybody being present — which is exactly the case the
`noUI` row above measures.

### What this chapter does not settle

`metadata/11-chapter.yaml` records that the guard is a **procedure over a serialised
command string**, and that the procedure assumes the need can be classified into one
of four kinds. A command that means `git push` without saying so defeats it. That is
not a bug in the guard; it is the boundary between a procedure and containment, and
chapter 42 is the chapter about containment.

## Sources, execution mode and limitations

**Execution mode:** **Run** — eight real sessions loading the chapter's guard and its retained pre-fix version, plus the chapter's own test file.

**Evidence scope:** `in_process_runtime`. The gate is extension code under test; the tool is Pi's `bash`; the model is scripted.

### What was read or run

- **Ran** `drivers/ch11-guard.ts`: eight sessions, each with its own temporary agent directory and working directory, loading `ch11-guard/guard.ts` and `ch11-guard/guard-original.ts` unmodified.
- **Checked the filesystem in every case** (`ran.flag`), so "blocked" means no effect rather than an error-shaped message.
- **Ran** `ch11-guard/guard.test.ts`.
- **Read** `examples/evidence.json`, chapter 11 rows (6 claims).

### Limitations

- **Commands really execute.** `git push` fails here because the temporary directory is not a repository. That is why the allowed cases are read from the flag file rather than from the push's own output.
- **A `UiRecorder` stands in for a person.** Its answers are queued by the driver. That is enough to observe *whether* a question was asked and what the answer decided; it is not a study of how a person answers.
- **No real model.** The scripted model asked for the commands; nothing here is evidence about whether a model would try to force push.
- **The pattern-matching exercise is Python**, restating the extension's own regex so the boundary can be inspected. The authoritative check that the guard blocks is the session run above and the chapter's tests.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
